# Synthetic Curriculum Ablation: Extra Level 2 Training

This run trains one continuous LoRA trajectory through Levels 1–4 with epoch schedule **3, 6, 3, 3**. It excludes Level 5 and writes checkpoints directly to a separate Google Drive directory. Rerunning the training cell resumes from its last saved checkpoint.

## 1. GPU and repository setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda} | available={torch.cuda.is_available()}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
REPO_DIR = Path("/content/table-cnn-mrc")
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print(f"Git commit: {commit}")
%cd /content/table-cnn-mrc

## 2. Install dependencies and mount Google Drive

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

## 3. Experiment configuration

In [ ]:
LEVELS = [1, 2, 3, 4]
EPOCHS_PER_LEVEL = [3, 6, 3, 3]
SEED = 42
BASE_MODEL = "Qwen/Qwen3-1.7B"
LEARNING_RATE = 5e-5
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
MAX_SEQUENCE_LENGTH = 2048
CHECKPOINT_EVERY_STEPS = 25

DRIVE_ROOT = Path("/content/drive/MyDrive/cnn_qwen_table_mcr")
CURRICULUM_OUTPUT = DRIVE_ROOT / "outputs/synthetic_curriculum_l2_6"
OFFICIAL_CACHE = DRIVE_ROOT / "outputs/diagnostics/wtq_official_1.0.2"
CURRICULUM_OUTPUT.mkdir(parents=True, exist_ok=True)

if len(LEVELS) != len(EPOCHS_PER_LEVEL):
    raise ValueError("LEVELS and EPOCHS_PER_LEVEL must have the same length")
for level in LEVELS:
    dataset_path = REPO_DIR / f"dataset_level_{level}.csv"
    if not dataset_path.is_file():
        raise FileNotFoundError(dataset_path)
print(f"Levels: {LEVELS}")
print(f"Epoch schedule: {EPOCHS_PER_LEVEL}")
print(f"Direct Drive output: {CURRICULUM_OUTPUT}")

## 4. Run Base → L1 (3) → L2 (6) → L3 (3) → L4 (3)

The base and every completed stage are evaluated on the full WTQ validation set. Do not reuse the old `synthetic_curriculum` output directory because its checkpoint signature belongs to a different schedule.

In [ ]:
command = [
    sys.executable, "-u", str(REPO_DIR / "scripts/run_curriculum.py"),
    "--data-root", str(REPO_DIR),
    "--output-dir", str(CURRICULUM_OUTPUT),
    "--official-cache-dir", str(OFFICIAL_CACHE),
    "--base-model", BASE_MODEL,
    "--levels", *[str(value) for value in LEVELS],
    "--epochs-per-level", *[str(value) for value in EPOCHS_PER_LEVEL],
    "--learning-rate", str(LEARNING_RATE),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
    "--max-sequence-length", str(MAX_SEQUENCE_LENGTH),
    "--checkpoint-every-steps", str(CHECKPOINT_EVERY_STEPS),
    "--seed", str(SEED),
]
command_text = " ".join(shlex.quote(str(part)) for part in command)
status_path = Path("/tmp/table_curriculum_l2_6_exit_code.txt")
status_path.unlink(missing_ok=True)
shell_command = (
    f"cd {shlex.quote(str(REPO_DIR))} && "
    f"PYTHONUNBUFFERED=1 TABLE_MRC_PLAIN_PROGRESS=1 TQDM_DISABLE=1 "
    f"{command_text}; printf '%s' $? > {shlex.quote(str(status_path))}"
)
get_ipython().system(shell_command)
if not status_path.is_file():
    raise RuntimeError("Training exit status was not recorded")
return_code = int(status_path.read_text(encoding="utf-8").strip())
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)

## 5. Compare results

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

results_path = CURRICULUM_OUTPUT / "results/curriculum_results.csv"
summary_path = CURRICULUM_OUTPUT / "results/curriculum_summary.json"
if not results_path.is_file():
    raise FileNotFoundError(results_path)
results = pd.read_csv(results_path)
display(results)

plt.figure(figsize=(8, 4))
plt.plot(results["stage"], results["wtq_validation_score"], marker="o")
plt.axhline(0.3582, color="gray", linestyle="--", label="previous best: 0.3582")
plt.ylabel("WTQ validation denotation accuracy")
plt.xlabel("Curriculum stage")
plt.title("Extra Level 2 curriculum ablation")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(summary, indent=2))